<a href="https://colab.research.google.com/github/ankita2002/LLMS/blob/main/HW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Homework 2 - From RNNs to Attention**

In this homework you will build a **sequence-to-sequence (seq2seq)** pipeline that converts dates from human-readable format (`DD/MM/YYYY`) to ISO format (`YYYY-MM-DD`).

Starting from scratch, you will generate the data, build the vocabulary, and progressively implement four models of increasing sophistication: a vanilla RNN, a weak LSTM with fixed gates, a full LSTM with learnable gates, and finally an LSTM augmented with dot-product attention.

By the end, you will be able to compare all four models side-by-side and see concretely what each architectural improvement contributes.

# Setup
Run the following cell first.

In [ ]:
#Imports
import torch.nn as nn
import random
import numpy as np
import torch

from datetime import date, timedelta
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Task 1: Data Preparation (4 Points)

Before we can train any model we need data. In this task you will:
1. **Generate date piars** – random `DD/MM/YYYY → YYYY-MM-DD` examples.
2. **Split** them into train / validation / test sets.
3. **Build vocabularies** (`src_stoi`, `src_itos`, `tgt_stoi`, `tgt_itos`) at the **character level**.
4. **Add special tokens**: `<pad>`, `<bos>` (beginning-of-sequence), `<eos>` (end-of-sequence).
5. **Encode** sequences to integer IDs and **pad** them to a fixed length.
6. **Wrap** everything in a PyTorch `Dataset` and create `DataLoader`s.

All subsequent tasks depend on `src_stoi`, `tgt_stoi`, `tgt_itos`, `train_loader`, and `val_loader`.So make sure those names are correct.

## Task 1.1: Date Pair Generation (0.5 Points)

Implement `generate_date_pairs(n)` that returns a list of `n` tuples `(src, tgt)` where:
- `src` is a date string in `DD/MM/YYYY` format (e.g. `'24/12/2025'`)
- `tgt` is the same date in `YYYY-MM-DD` format (e.g. `'2025-12-24'`)

Make sure you only generate **valid** calendar dates (e.g. no 30 February).

```
generate_date_pairs(3)
# Example output:
# [('07/03/2011', '2011-03-07'), ('19/11/1998', '1998-11-19'), ...]
```

In [ ]:
def generate_date_pairs(n: int):
    # TODO: Generate n random (src, tgt) date string pairs

    raise NotImplementedError


In [ ]:
#Sanity check
samples = generate_date_pairs(5)
for src, tgt in samples:
    print(src, "=>", tgt)

## Task 1.2: Train / Val / Test Split (0.5 Points)

Generate **10 000** date pairs in total and split them:


Suggested Split:
- **train_pairs**: first 8 000
- **val_pairs**: next 1 000
- **test_pairs**: last 1 000

Store them in variables named exactly `train_pairs`, `val_pairs`, and `test_pairs`.

In [ ]:
all_pairs = generate_date_pairs(10_000)

#TODO: Implement the train, val, and test split
train_pairs = ...
val_pairs   = ...
test_pairs  = ...


In [ ]:
#Sanity check
print(f"Train: {len(train_pairs)}  Val: {len(val_pairs)}  Test: {len(test_pairs)}")
#expected: Train: 8000  Val: 1000  Test: 1000

## Task 1.3: Vocabulary Building (1 Point)

Build **character-level** vocabularies for the source side (`DD/MM/YYYY`) and the target side (`YYYY-MM-DD`).

Each vocabulary must include the three special tokens **`<pad>`**, **`<bos>`**, **`<eos>`** — and they must occupy indices **0, 1, 2** respectively.

You must produce four mappings:
- `src_stoi` – source char → index
- `src_itos` – index → source char
- `tgt_stoi` – target char → index
- `tgt_itos` – index → target char

**Hint:** The source alphabet is `{0-9, /}` and the target alphabet is `{0-9, -}`.

In [ ]:
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>"]
PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2

#TODO: Collect all unique characters that appear in source strnigs
src_chars = ...

#TODO: src_stoi: special tokens first (at indices 0,1,2), then sorted chars
src_stoi = ...

#TODO: Build src_itos as the inverse mapping
src_itos = ...

#TODO: Collect all unique characters that appear in target strings
# Your code here
tgt_chars = ...

#TODO: Build tgt_stoi and tgt_itos
tgt_stoi = ...

tgt_itos = ...



In [ ]:
# Sanity Check
# expected:
# Source vocab: {'<pad>': 0, '<bos>': 1, '<eos>': 2, '/': 3, '0': 4, '1': 5, '2': 6, '3': 7, '4': 8, '5': 9, '6': 10, '7': 11, '8': 12, '9': 13}
# Target vocab: {'<pad>': 0, '<bos>': 1, '<eos>': 2, '-': 3, '0': 4, '1': 5, '2': 6, '3': 7, '4': 8, '5': 9, '6': 10, '7': 11, '8': 12, '9': 13}
print("Source vocab:", src_stoi)
print("Target vocab:", tgt_stoi)

## Task 1.4: Encoding Helpers (1 Point)

Implement two helper functions used both during training and at inference time.

**`encode_source(text, stoi)`** — turns a raw source string into a list of integer IDs with `<eos>` at the end.

Example: '24/12/2025' -> [3, 5, 10, 2, 3, ..., EOS_IDX]

**`encode_target(text, stoi)`** — turns a raw target string into a list of integer IDs **wrapped** with `<bos>` at the start and `<eos>` at the end.

Example: '2025-12-24' -> [BOS_IDX, 4, 1, 3, ...., EOS_IDX]

**`ids_to_text(ids, itos)`** — converts a list of IDs back to a string, skipping `<bos>`, `<eos>`, and `<pad>` tokens.

In [ ]:
def encode_source(text: str, stoi: dict) -> list:
    #TODO:
    raise NotImplementedError


def encode_target(text: str, stoi: dict) -> list:
    #TODO:
    raise NotImplementedError


def ids_to_text(ids: list, itos: dict) -> str:
    #TODO:
    raise NotImplementedError



In [ ]:
# Sanity check
# Example Output:
# Source: 28/01/1950 -> [6, 12, 3, 4, 5, 3, 5, 13, 9, 4, 2]
# Target: 1950-01-28 -> [1, 5, 13, 9, 4, 3, 4, 5, 3, 6, 12, 2]
# Decoded: 1950-01-28
src_ex, tgt_ex = train_pairs[0]
print("Source:", src_ex, "->", encode_source(src_ex, src_stoi))
print("Target:", tgt_ex, "->", encode_target(tgt_ex, tgt_stoi))
print("Decoded:", ids_to_text(encode_target(tgt_ex, tgt_stoi), tgt_itos))

## Task 1.5: Dataset and DataLoaders (1 Point)

Implement a PyTorch `Dataset` called `DateDataset` and create `train_loader` and `val_loader`.

Each item returned by `__getitem__` must be a tuple `(src_tensor, tgt_tensor)` where both are `torch.long` tensors **padded to a fixed length** using `PAD_IDX`.

Use a **batch size of 64** and **shuffle the training loader**. Do not shuffle the validation loader.

**Hint:** The maximum source length is 11 characters. The maximum target length (including `<bos>` and `<eos>`) is 12.

In [ ]:
SRC_MAX_LEN = 11  # DD/MM/YYYY is always 10 characters + <eos>
TGT_MAX_LEN = 12  # <bos> + YYYY-MM-DD (10 chars) + <eos>


class DateDataset(Dataset):
    def __init__(self, pairs, src_stoi, tgt_stoi,
                 src_max_len=SRC_MAX_LEN, tgt_max_len=TGT_MAX_LEN):
        #TODO:
        raise NotImplementedError

    def __len__(self):
        #TODO:
        raise NotImplementedError

    def __getitem__(self, idx):
        #TODO:
        raise NotImplementedError

train_dataset = DateDataset(train_pairs, src_stoi, tgt_stoi)
val_dataset   = DateDataset(val_pairs,   src_stoi, tgt_stoi)

#TODO: Create DataLoaders
# use batch_size=64, shuffle train, don't shuffle val
train_loader = ...
val_loader   = ...


In [ ]:
# Sanity check
src_batch, tgt_batch = next(iter(train_loader))
print("src batch shape:", src_batch.shape)   # expected: (64, 11)
print("tgt batch shape:", tgt_batch.shape)   # expected: (64, 12)

# Task 2: RNN baseline  (3.5 Points)
In this task, you implement a basic sequence-to-sequence model using vanilla RNNs.

The model consists of two parts:

1. **Encoder**  
   The encoder reads the full source sequence and converts it into hidden states.  
   It returns:
   - `encoder_outputs`: hidden states for all source positions, shape `[batch_size, src_len, hidden_dim]`
   - `hidden`: the final hidden state, shape `[1, batch_size, hidden_dim]`

2. **Decoder**  
   The decoder generates the target sequence one token at a time.  
   At each step, it receives:
   - the previous target token `input_tok`, shape `[batch_size]`
   - the previous hidden state `hidden`

   It returns:
   - `logits`: raw vocabulary scores for the next token, shape `[batch_size, output_dim]`
   - the updated hidden state

In [ ]:
class RNNEncoder(nn.Module):
  def __init__(self, input_dim, emb_dim, hid_dim, pad_idx):
    super().__init__()
    self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=pad_idx)
    self.rnn = nn.RNN(emb_dim, hid_dim, batch_first=True)

  def forward(self, src):
    # src: [batch_size, src_len]
    # TODO: embed the source sequence
    embedded = ...
    # TODO: pass embeddings through the RNN
    encoder_outputs = ...
    # TODO: return encoder outputs and final hidden state
    raise NotImplementedError

class RNNDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(emb_dim, hid_dim, batch_first=True)
        self.fc_out = nn.Linear(hid_dim, output_dim)

    def forward(self, input_tok, hidden):
      # input_tok: [batch_size]
      # hidden: [1, batch_size, hidden_dim]
      # TODO: embed input_tok
      # TODO: unsqueeze to length-1 sequence
      embedded = ...
      # TODO: run one RNN step
      output, hidden =  ...
      # TODO: project hidden state to vocabulary logits
      logits = ...
      raise NotImplementedError

class Seq2SeqRNN(nn.Module):
  def __init__(self, encoder, decoder):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder

  def forward(self, src, tgt, teacher_forcing_ratio=0.5):
    batch_size, tgt_len = tgt.shape
    vocab_size = self.decoder.fc_out.out_features
    outputs = torch.zeros(batch_size, tgt_len - 1, vocab_size, device=src.device)

    # TODO: run the encoder
    encoder_outputs, hidden = ...
    # TODO: initialize decoder input with <bos>
    input_tok = ...
    # TODO: autoregressive decoder loop with teacher forcing

    raise NotImplementedError

## Training and evaluation helpers

In this task, you implement the training and evaluation logic for the sequence-to-sequence model.

### Sequence Prediction Setup

The model predicts the target sequence **one step ahead**:

- The decoder receives `<bos>` as the first input
- It predicts tokens for positions `1 ... T-1`

### Cross-Entropy Loss

We use `nn.CrossEntropyLoss`, which expects:

- predictions of shape: `[N, vocab_size]`
- targets of shape: `[N]`

So you must **flatten both tensors**:

- predictions → `[batch_size * (tgt_len - 1), vocab_size]`
- targets → `[batch_size * (tgt_len - 1)]`

The loss should ignore padding tokens.

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

def train_one_epoch(model, loader, optimizer, clip=1.0, teacher_forcing_ratio=0.5):
  model.train()
  total_loss = 0.0


  for src, tgt in loader:
    src, tgt = src.to(device), tgt.to(device)
    optimizer.zero_grad()

    output = model(src, tgt, teacher_forcing_ratio=teacher_forcing_ratio)
    if isinstance(output, tuple):
      output = output[0]

    #output: [batch_size, tgt_len-1, vocab_size]
    #tgt[:, 1:]: [batch_size, tgt_len-1]
    # TODO: compute sequence cross-entropy loss

    raise NotImplementedError

  return total_loss/len(loader)

@torch.no_grad()
def evaluate_model(model, loader):
  model.eval()
  total_loss = 0.0

  for src, tgt in loader:
    src, tgt = src.to(device), tgt.to(device)
    output = model(src, tgt, teacher_forcing_ratio=0.0)
    if isinstance(output, tuple):
      output = output[0]

    # TODO: compute validation loss without teacher forcing
    raise NotImplementedError

  return total_loss / len(loader)

In [ ]:
#Sanity check
RNN_EMB_DIM = 32
RNN_HID_DIM = 64

rnn_model = Seq2SeqRNN(
    RNNEncoder(len(src_stoi), RNN_EMB_DIM, RNN_HID_DIM, src_stoi["<pad>"]),
    RNNDecoder(len(tgt_stoi), RNN_EMB_DIM, RNN_HID_DIM, tgt_stoi["<pad>"]),
).to(device)

optimizer = torch.optim.Adam(rnn_model.parameters(), lr=1e-3)

for epoch in range(5):
  train_loss = train_one_epoch(rnn_model, train_loader, optimizer)
  val_loss = evaluate_model(rnn_model, val_loader)
  print(f"[RNN] Epoch {epoch+1}: train={train_loss:.4f} val={val_loss:.4f}")

# Task 3: Hard-gated LSTM (2.5 Points)
In this task, you implement a simplified LSTM-like encoder-decoder model.

The goal is to introduce the idea of an LSTM memory cell.

Unlike the vanilla RNN, this model keeps two states:

- `hidden_state`: the short-term state used for prediction
- `cell_state`: the long-term memory state

We use three gates:

- `f_t = 0.5`: forget gate

- `i_t = 0.5`: input gate  

- `o_t = 1.0`: output gate

In [ ]:
class WeakLSTMEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=pad_idx)
        self.x_proj = nn.Linear(emb_dim, hid_dim)
        self.h_proj = nn.Linear(hid_dim, hid_dim)

    def forward(self, src):
        emb = self.embedding(src)
        batch_size, src_len, _ = emb.shape

        hidden_state = torch.zeros(batch_size, self.h_proj.out_features, device=src.device)
        cell_state = torch.zeros_like(hidden_state)
        outputs = []

        f_t = 0.5
        i_t = 0.5
        o_t = 1.0

        for t in range(src_len):
            x_t = emb[:, t, :]

            # TODO: compute candidate state g_t
            # TODO: update c using fixed gates
            # TODO: update h using c and o_t
            # TODO: store h in outputs
            raise NotImplementedError

        outputs = torch.stack(outputs, dim=1)
        return outputs, (hidden_state.unsqueeze(0), cell_state.unsqueeze(0))

class WeakLSTMDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=pad_idx)
        self.x_proj = nn.Linear(emb_dim, hid_dim)
        self.h_proj = nn.Linear(hid_dim, hid_dim)
        self.fc_out = nn.Linear(hid_dim, output_dim)

    def forward(self, input_tok, hidden, cell):
        emb = self.embedding(input_tok)

        f_t = 0.5
        i_t = 0.5
        o_t = 1.0

        h_prev = hidden.squeeze(0)
        c_prev = cell.squeeze(0)

        # TODO: compute candidate state
        # TODO: update c_t
        # TODO: update h_t
        # TODO: project h_t to logits and restore [1,B,H] shape
        raise NotImplementedError

class Seq2SeqWeakLSTM(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        vocab_size = self.decoder.fc_out.out_features
        outputs = torch.zeros(batch_size, tgt_len - 1, vocab_size, device=src.device)

        encoder_outputs, (hidden, cell) = self.encoder(src)
        input_tok = tgt[:, 0]

        for t in range(1, tgt_len):
          # TODO: run the weak LSTM decoder for one step
          # TODO: store logits
          # TODO: teacher forcing update
          raise NotImplementedError

        return outputs

In [ ]:
#Sanity Check
WEAK_EMB_DIM = 32
WEAK_HID_DIM = 32

weak_lstm_model = Seq2SeqWeakLSTM(
    WeakLSTMEncoder(len(src_stoi), WEAK_EMB_DIM, WEAK_HID_DIM, src_stoi["<pad>"]),
    WeakLSTMDecoder(len(tgt_stoi), WEAK_EMB_DIM, WEAK_HID_DIM, tgt_stoi["<pad>"]),
).to(device)

optimizer = torch.optim.Adam(weak_lstm_model.parameters(), lr=1e-3)

for epoch in range(5):
    train_loss = train_one_epoch(weak_lstm_model, train_loader, optimizer)
    val_loss = evaluate_model(weak_lstm_model, val_loader)
    print(f"[Weak LSTM] Epoch {epoch+1}: train={train_loss:.4f} val={val_loss:.4f}")

# Task 4: Soft-gated LSTM (3 Points)
In the previous task, the LSTM-like model used fixed gates such as `0.5` and `1.0`.  

In this task, you implement a stronger LSTM where the gates are **learnable**.

At each time step, the model uses:

- the current input embedding
- the previous hidden state

These are concatenated into:

`gate_input = [input_t ; hidden_state]`

The model then computes four values:

- `f_t`: forget gate  
- `i_t`: input gate  
- `o_t`: output gate  
- `g_t`: candidate memory content  

In [ ]:
class LearnableLSTMEncoder(nn.Module):
  def __init__(self, input_dim, emb_dim, hid_dim, pad_idx):
    super().__init__()
    self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=pad_idx)
    self.f_gate = nn.Linear(emb_dim + hid_dim, hid_dim)
    self.i_gate = nn.Linear(emb_dim + hid_dim, hid_dim)
    self.o_gate = nn.Linear(emb_dim + hid_dim, hid_dim)
    self.g_gate = nn.Linear(emb_dim + hid_dim, hid_dim)

  def forward(self, src):
    emb = self.embedding(src)
    batch_size, src_len, _ = emb.shape
    hidden_dim = self.f_gate.out_features

    hidden_state = torch.zeros(batch_size, hidden_dim, device=src.device)
    cell_state = torch.zeros(batch_size, hidden_dim, device=src.device)
    outputs = []

    for t in range(src_len):
      input_t = emb[:, t, :]
      gate_input = torch.cat([input_t, hidden_state], dim=-1)
      # TODO: compute f_t, i_t, o_t with sigmoid
      # TODO: compute g_t with tanh
      # TODO: update cell_state and hidden_state
      # TODO: store hidden_state
      raise NotImplementedError

    outputs = torch.stack(outputs, dim=1)
    return outputs, (hidden_state.unsqueeze(0), cell_state.unsqueeze(0))

class LearnableLSTMDecoder(nn.Module):
  def __init__(self, output_dim, emb_dim, hid_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=pad_idx)
        self.f_gate = nn.Linear(emb_dim + hid_dim, hid_dim)
        self.i_gate = nn.Linear(emb_dim + hid_dim, hid_dim)
        self.o_gate = nn.Linear(emb_dim + hid_dim, hid_dim)
        self.g_gate = nn.Linear(emb_dim + hid_dim, hid_dim)
        self.fc_out = nn.Linear(hid_dim, output_dim)

  def forward(self, input_tok, hidden, cell):
    emb = self.embedding(input_tok)
    h_prev = hidden.squeeze(0)
    c_prev = cell.squeeze(0)
    gate_input = torch.cat([emb, h_prev], dim=-1)

    # TODO: compute learnable gates
    # TODO: update cell state and hidden state
    # TODO: project to logits
    raise NotImplementedError

class Seq2SeqLearnableLSTM(nn.Module):
  def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

  def forward(self, src, tgt, teacher_forcing_ratio=0.5):
    batch_size, tgt_len = tgt.shape
    vocab_size = self.decoder.fc_out.out_features
    outputs = torch.zeros(batch_size, tgt_len - 1, vocab_size, device=src.device)

    encoder_outputs, (hidden, cell) = self.encoder(src)
    input_token = tgt[:, 0]

    for t in range(1, tgt_len):
      logits, hidden, cell = self.decoder(input_token, hidden, cell)
      outputs[:, t - 1] = logits

      teacher_force = random.random() < teacher_forcing_ratio
      top1 = logits.argmax(dim=1)
      input_token = tgt[:, t] if teacher_force else top1

    return outputs

In [ ]:
#Sanity check
LSTM_EMB_DIM = 32
LSTM_HID_DIM = 64

learnable_lstm_model = Seq2SeqLearnableLSTM(
    LearnableLSTMEncoder(len(src_stoi), LSTM_EMB_DIM, LSTM_HID_DIM, src_stoi["<pad>"]),
    LearnableLSTMDecoder(len(tgt_stoi), LSTM_EMB_DIM, LSTM_HID_DIM, tgt_stoi["<pad>"]),
).to(device)

optimizer = torch.optim.Adam(learnable_lstm_model.parameters(), lr=1e-3)

for epoch in range(5):
    train_loss = train_one_epoch(learnable_lstm_model, train_loader, optimizer)
    val_loss = evaluate_model(learnable_lstm_model, val_loader)
    print(f"[Learnable LSTM] Epoch {epoch+1}: train={train_loss:.4f} val={val_loss:.4f}")

# Task 5: Add attention   (3 Points)
So far, the decoder only received the final hidden and cell states from the encoder.

This creates a bottleneck

In this task, you add an attention mechanism.

### Dot-Product Attention

The attention module receives:

- `query`
- `encoder_outputs`

It should compute:

1. **Attention scores**  

2. **Attention weights**  

3. **Context vector**

In [ ]:
class DotAttention(nn.Module):
  def forward(self, query, encoder_outputs):
    # query: [batch_size, hidden_dim]
    # encoder_outputs: [batch_size, src_len, hidden_dim]
    # TODO: compute dot-product attention scores
    # TODO: normalize scores with softmax
    # TODO: compute context vector as weighted sum of encoder outputs
    raise NotImplementedError

class AttnLearnableLSTMDecoder(nn.Module):
  def __init__(self, output_dim, emb_dim, hid_dim, pad_idx):
    super().__init__()
    self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=pad_idx)
    self.attention = DotAttention()
    self.f_gate = nn.Linear(emb_dim + hid_dim + hid_dim, hid_dim)
    self.i_gate = nn.Linear(emb_dim + hid_dim + hid_dim, hid_dim)
    self.o_gate = nn.Linear(emb_dim + hid_dim + hid_dim, hid_dim)
    self.g_gate = nn.Linear(emb_dim + hid_dim + hid_dim, hid_dim)
    self.fc_out = nn.Linear(hid_dim + hid_dim, output_dim)

  def forward(self, input_tok, hidden, cell, encoder_outputs):
    emb = self.embedding(input_tok)
    h_prev = hidden.squeeze(0)
    c_prev = cell.squeeze(0)

    # TODO: use current decoder state as attention query
    # TODO: get context and attention weights
    # TODO: concatenate embedding, previous hidden state, and context
    # TODO: compute learnable gates
    # TODO: update c_t and h_t
    # TODO: project [h_t ; context] to logits
    raise NotImplementedError

class Seq2SeqLSTMAttn(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
      batch_size, tgt_len = tgt.shape
      vocab_size = self.decoder.fc_out.out_features
      outputs = torch.zeros(batch_size, tgt_len - 1, vocab_size, device = src.device)
      all_attn = []

      encoder_outputs, (hidden, cell) = self.encoder(src)
      input_tok = tgt[:, 0]

      for t in range(1, tgt_len):
        # TODO: decoder step with attention
        # TODO: store logits and attention weights
        # TODO: teacher forcing update
        raise NotImplementedError

      all_attn = torch.stack(all_attn, dim=1)
      return outputs, all_attn


In [ ]:
#Sanity check
attn_model = Seq2SeqLSTMAttn(
    LearnableLSTMEncoder(len(src_stoi), LSTM_EMB_DIM, LSTM_HID_DIM, src_stoi["<pad>"]),
    AttnLearnableLSTMDecoder(len(tgt_stoi), LSTM_EMB_DIM, LSTM_HID_DIM, tgt_stoi["<pad>"]),
).to(device)

optimizer = torch.optim.Adam(attn_model.parameters(), lr=1e-3)

for epoch in range(5):
    train_loss = train_one_epoch(attn_model, train_loader, optimizer)
    val_loss = evaluate_model(attn_model, val_loader)
    print(f"[LSTM + Attention] Epoch {epoch+1}: train={train_loss:.4f} val={val_loss:.4f}")

# Task 6: Inference  (1 Point)
At inference time we don't have the ground-truth target sequence, so we generate one token at a time.


In [ ]:
@torch.no_grad()
def greedy_decode(model, src_text, src_stoi, tgt_stoi, tgt_itos, max_len=16, attention=False):
    model.eval()

    src_ids = encode_source(src_text, src_stoi)
    src = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(0)

    generated = [tgt_stoi["<bos>"]]
    attn_map = []

    if attention:
        encoder_outputs, (hidden, cell) = model.encoder(src)
    else:
        encoder_outputs, hidden_or_state = model.encoder(src)

    input_tok = torch.tensor([tgt_stoi["<bos>"]], device=device)

    for _ in range(max_len):
      if attention:
        # TODO: run attention decoder and collect attn_weights
        raise NotImplementedError
      else:
          if isinstance(model, Seq2SeqRNN):
            # TODO: one decoding step for the RNN model
            raise NotImplementedError
          else:
              hidden, cell = hidden_or_state
              # TODO: one decoding step for the LSTM-style models
              raise NotImplementedError

      # TODO: greedy next-token choice
      # TODO: stop on <eos>
      # TODO: feed next_tok back into the decoder
      raise NotImplementedError

    decoded = ids_to_text(generated, tgt_itos)
    return decoded, attn_map



## Sanity check: Model Comparison
Use the following cell to compare predictions from all four models.

In [ ]:
examples = [
    "01/01/2001",
    "24/12/2025",
    "13/05/1999",
    "07/11/2018",
]

for src_text in examples:
    pred_rnn, _ = greedy_decode(rnn_model, src_text, src_stoi, tgt_stoi, tgt_itos, attention=False)
    pred_weak, _ = greedy_decode(weak_lstm_model, src_text, src_stoi, tgt_stoi, tgt_itos, attention=False)
    pred_lstm, _ = greedy_decode(learnable_lstm_model, src_text, src_stoi, tgt_stoi, tgt_itos, attention=False)
    pred_attn, attn = greedy_decode(attn_model, src_text, src_stoi, tgt_stoi, tgt_itos, attention=True)

    print(f"Source:      {src_text}")
    print(f"Target:      {src_text[6:10]}-{src_text[3:5]}-{src_text[0:2]}")
    print(f"RNN:         {pred_rnn}")
    print(f"Weak LSTM:   {pred_weak}")
    print(f"Learn. LSTM: {pred_lstm}")
    print(f"LSTM+Attn:   {pred_attn}")
    print("-" * 50)

# Task 7: Multiple Choice Questions (3 Points)

For each question, print your answer(s) in the code cell below it.

**Example:**
```python
print("A")
```




### Q1 (0.5 pts)

We append `<eos>` to the **source** sequence and wrap the **target** with `<bos>` ... `<eos>`. What is the main purpose of the `<eos>` token on the source side?

A. It tells the encoder to stop updating its hidden state after the last real character.  
B. It replaces the `<pad>` token so that only one special token is needed.  
C. It pads the source sequence to a fixed length.  
D. It acts as an explicit end-of-input signal that the encoder can learn to recognise, after which the final hidden state is a clean summary of the whole sequence.  

In [ ]:
# Write your answer below
print("")


### Q2 (0.5 pts)

We pass `ignore_index=PAD_IDX` to `nn.CrossEntropyLoss`. What is the direct effect of this setting?

A. Padding tokens are replaced with zeros before the loss is computed.  
B. The loss contribution from padding positions is set to zero, so they do not affect the gradients.  
C. The model learns to predict `<pad>` with higher confidence.  
D. It reduces the effective vocabulary size during training.



In [ ]:
# Write your answer below
print("")

### Q3 (0.5 pt)

What is the main problem that the LSTM was designed to solve compared to a vanilla RNN?

A. RNNs suffer from vanishing gradients, making it hard to learn long-range dependencies; the LSTM cell state provides a path for gradients to flow across many steps with minimal attenuation.  
B. RNNs cannot handle variable-length sequences, while LSTMs can.  
C. RNNs require more memory than LSTMs for the same hidden dimension.  
D. RNNs can only process sequences in one direction, while LSTMs are bidirectional by default.






In [ ]:
# Write your answer below
print("")

### Q4 (0.5 pts)

In the Weak LSTM the forget gate is fixed at `f = 0.5`. What does this mean for information in the cell state?

A. Exactly half of the cell-state memory is discarded at every time step, regardless of the input.  
B. The model learns to forget the 50 % of information that is least useful.  
C. The cell state is reset to zero every two time steps.  
D. The hidden state and cell state become identical after many steps.


In [ ]:
# Write your answer below
print("")

### Q5 (0.5 pts)

During training, teacher forcing feeds the ground-truth target token as the next decoder input instead of the model's own prediction. Which of the following is a known **disadvantage** of teacher forcing?

A. It makes training slower because the decoder must run twice per step.  
B. It can create a train-inference mismatch: at inference time the model must use its own (potentially wrong) predictions, which it was never trained to recover from.  
C. It prevents the model from learning to generate `<eos>` correctly.  
D. It increases the loss because gold tokens are harder to predict than sampled tokens.



In [ ]:
# Write your answer below
print("")

### Q6 (0.5 pts)


In dot-product attention, how is the context vector produced at each decoder step?

A. It is the final encoder hidden state, passed unchanged to the decoder.  
B. It is the average of all encoder outputs, weighted equally.        
C. It is a weighted sum of all encoder outputs, where the weights come from softmax-normalising the dot products between the decoder's hidden state and each encoder output.  
D. It is computed by a separate feedforward network trained independently from the seq2seq model.

In [ ]:
# Write your answer below
print("")